In [8]:
import numpy as np
import pandas as pd
import warnings
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, MinMaxScaler

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────
# 1. LOAD DATA
# ─────────────────────────────────────────────
df = pd.read_csv("Titanic-Dataset.csv")

# ─────────────────────────────────────────────
# 2. DROP IRRELEVANT COLUMNS
# ─────────────────────────────────────────────
df.drop(columns=['Cabin', 'Ticket', 'Name', 'PassengerId'], inplace=True)

# ─────────────────────────────────────────────
# 3. HANDLE MISSING VALUES
# ─────────────────────────────────────────────
# Embarked: fill with mode
df['Embarked'].fillna('S', inplace=True)

# Fare: fill with median
df['Fare'].fillna(df['Fare'].median(), inplace=True)

# Age: fill with median grouped by Sex and Pclass
df['Age'] = df.groupby(['Sex', 'Pclass'])['Age'].transform(
    lambda x: x.fillna(x.median())
)
df['Age'] = df['Age'].astype('int64')

# ─────────────────────────────────────────────
# 4. FEATURE ENGINEERING
# ─────────────────────────────────────────────

# --- Title extraction from Name (re-read before drop for title extraction) ---
raw_df = pd.read_csv("Titanic-Dataset.csv")
df['Title'] = raw_df['Name'].str.split(", ", expand=True)[1].str.split(".", expand=True)[0]

# Consolidate rare titles
df['Title'] = df['Title'].replace(
    ['Lady', 'the Countess', 'Capt', 'Col', 'Don', 'Dr',
     'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare'
)
df['Title'] = df['Title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})

# --- Family Size ---
df['Family_size'] = raw_df['SibSp'] + raw_df['Parch'] + 1

def categorize_family(n):
    if n == 1:
        return 'Alone'
    elif n < 5:
        return 'Small'
    else:
        return 'Large'

df['Family_size'] = df['Family_size'].apply(categorize_family)

# Drop original SibSp and Parch (now encoded in Family_size)
df.drop(columns=['SibSp', 'Parch'], inplace=True)

print("Columns before encoding:", df.columns.tolist())
print("Shape before encoding:", df.shape)
print(df.head())

# ─────────────────────────────────────────────
# 5. SEPARATE TARGET & FEATURES
# ─────────────────────────────────────────────
# Keep 'Survived' aside — we're only preprocessing features
survived = df['Survived'] if 'Survived' in df.columns else None
X = df.drop(columns=['Survived'], errors='ignore')

# ─────────────────────────────────────────────
# 6. ENCODE & SCALE
# ─────────────────────────────────────────────
# Column definitions
num_cols     = ['Age', 'Fare']
onehot_cols  = ['Embarked', 'Title', 'Family_size']
ordinal_cols = ['Sex']          # male=1, female=0 via OrdinalEncoder
passthru_cols = ['Pclass']      # already numeric ordinal (1, 2, 3)

preprocessor = ColumnTransformer([
    ('scaling',  MinMaxScaler(),                        num_cols),
    ('onehot',   OneHotEncoder(sparse_output=False),    onehot_cols),
    ('ordinal',  OrdinalEncoder(),                      ordinal_cols),
], remainder='passthrough')   # passes Pclass through unchanged

X_transformed = preprocessor.fit_transform(X)

# ─────────────────────────────────────────────
# 7. REBUILD DATAFRAME WITH COLUMN NAMES
# ─────────────────────────────────────────────
onehot_feature_names = preprocessor.named_transformers_['onehot'].get_feature_names_out(onehot_cols).tolist()

all_columns = (
    num_cols +
    onehot_feature_names +
    ordinal_cols +
    [c for c in X.columns if c not in num_cols + onehot_cols + ordinal_cols]  # remainder
)

preprocessedDB = pd.DataFrame(X_transformed, columns=all_columns)

# Re-attach target column if it existed
if survived is not None:
    preprocessedDB.insert(0, 'Survived', survived.values)

# ─────────────────────────────────────────────
# 8. SAVE
# ─────────────────────────────────────────────
preprocessedDB.to_csv("preprocessedDB.csv", index=False)

print("\n✅ Preprocessing complete!")
print("Shape of preprocessedDB:", preprocessedDB.shape)
print(preprocessedDB.head())

Columns before encoding: ['Survived', 'Pclass', 'Sex', 'Age', 'Fare', 'Embarked', 'Title', 'Family_size']
Shape before encoding: (891, 8)
   Survived  Pclass     Sex  Age     Fare Embarked Title Family_size
0         0       3    male   22   7.2500        S    Mr       Small
1         1       1  female   38  71.2833        C   Mrs       Small
2         1       3  female   26   7.9250        S  Miss       Alone
3         1       1  female   35  53.1000        S   Mrs       Small
4         0       3    male   35   8.0500        S    Mr       Alone

✅ Preprocessing complete!
Shape of preprocessedDB: (891, 17)
   Survived     Age      Fare  Embarked_C  Embarked_Q  Embarked_S  \
0         0  0.2750  0.014151         0.0         0.0         1.0   
1         1  0.4750  0.139136         1.0         0.0         0.0   
2         1  0.3250  0.015469         0.0         0.0         1.0   
3         1  0.4375  0.103644         0.0         0.0         1.0   
4         0  0.4375  0.015713         0.0

In [9]:
import numpy as np
import pandas as pd
import warnings
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, roc_auc_score
)

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────
# 1. LOAD PREPROCESSED DATA
# ─────────────────────────────────────────────
df = pd.read_csv("preprocessedDB.csv")

# ─────────────────────────────────────────────
# 2. SPLIT FEATURES & TARGET
# ─────────────────────────────────────────────
X = df.drop(columns=['Survived'])
y = df['Survived']

# ─────────────────────────────────────────────
# 3. TRAIN / TEST SPLIT
# ─────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size : {X_train.shape[0]} samples")
print(f"Test  size : {X_test.shape[0]} samples")

# ─────────────────────────────────────────────
# 4. TRAIN LOGISTIC REGRESSION
# ─────────────────────────────────────────────
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

# ─────────────────────────────────────────────
# 5. EVALUATE ON TEST SET
# ─────────────────────────────────────────────
y_pred      = model.predict(X_test)
y_pred_prob = model.predict_proba(X_test)[:, 1]

accuracy  = accuracy_score(y_test, y_pred)
roc_auc   = roc_auc_score(y_test, y_pred_prob)
cv_scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')

print("\n" + "="*45)
print("         LOGISTIC REGRESSION RESULTS")
print("="*45)
print(f"  Test  Accuracy  : {accuracy:.4f}  ({accuracy*100:.2f}%)")
print(f"  ROC-AUC Score   : {roc_auc:.4f}")
print(f"  CV Accuracy     : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}  (5-fold)")
print("="*45)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Not Survived', 'Survived']))

print("Confusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(
    cm,
    index   = ['Actual: Not Survived', 'Actual: Survived'],
    columns = ['Pred: Not Survived',   'Pred: Survived']
)
print(cm_df)

# ─────────────────────────────────────────────
# 6. FEATURE IMPORTANCE (coefficients)
# ─────────────────────────────────────────────
coef_df = pd.DataFrame({
    'Feature'    : X.columns,
    'Coefficient': model.coef_[0]
}).sort_values('Coefficient', ascending=False)

print("\nTop Positive Predictors (→ Survived):")
print(coef_df.head(5).to_string(index=False))

print("\nTop Negative Predictors (→ Not Survived):")
print(coef_df.tail(5).to_string(index=False))

Train size : 712 samples
Test  size : 179 samples

         LOGISTIC REGRESSION RESULTS
  Test  Accuracy  : 0.8380  (83.80%)
  ROC-AUC Score   : 0.8776
  CV Accuracy     : 0.8249 ± 0.0197  (5-fold)

Classification Report:
              precision    recall  f1-score   support

Not Survived       0.85      0.89      0.87       110
    Survived       0.81      0.75      0.78        69

    accuracy                           0.84       179
   macro avg       0.83      0.82      0.83       179
weighted avg       0.84      0.84      0.84       179

Confusion Matrix:
                      Pred: Not Survived  Pred: Survived
Actual: Not Survived                  98              12
Actual: Survived                      17              52

Top Positive Predictors (→ Survived):
          Feature  Coefficient
     Title_Master     1.344292
Family_size_Alone     0.720128
Family_size_Small     0.656502
        Title_Mrs     0.618514
             Fare     0.541932

Top Negative Predictors (→ Not Survi

In [10]:
import numpy as np
import pandas as pd
import warnings
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, roc_auc_score
)

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────
# 1. LOAD PREPROCESSED DATA
# ─────────────────────────────────────────────
df = pd.read_csv("preprocessedDB.csv")

# ─────────────────────────────────────────────
# 2. SPLIT FEATURES & TARGET
# ─────────────────────────────────────────────
X = df.drop(columns=['Survived'])
y = df['Survived']

# ─────────────────────────────────────────────
# 3. TRAIN / TEST SPLIT
# ─────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size : {X_train.shape[0]} samples")
print(f"Test  size : {X_test.shape[0]} samples")

# ─────────────────────────────────────────────
# 4. TRAIN LOGISTIC REGRESSION
# ─────────────────────────────────────────────
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

# ─────────────────────────────────────────────
# 5. EVALUATE ON TEST SET
# ─────────────────────────────────────────────
y_pred      = model.predict(X_test)
y_pred_prob = model.predict_proba(X_test)[:, 1]

accuracy  = accuracy_score(y_test, y_pred)
roc_auc   = roc_auc_score(y_test, y_pred_prob)
cv_scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')

print("\n" + "="*45)
print("         LOGISTIC REGRESSION RESULTS")
print("="*45)
print(f"  Test  Accuracy  : {accuracy:.4f}  ({accuracy*100:.2f}%)")
print(f"  ROC-AUC Score   : {roc_auc:.4f}")
print(f"  CV Accuracy     : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}  (5-fold)")
print("="*45)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Not Survived', 'Survived']))

print("Confusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(
    cm,
    index   = ['Actual: Not Survived', 'Actual: Survived'],
    columns = ['Pred: Not Survived',   'Pred: Survived']
)
print(cm_df)

# ─────────────────────────────────────────────
# 6. FEATURE IMPORTANCE (coefficients)
# ─────────────────────────────────────────────
coef_df = pd.DataFrame({
    'Feature'    : X.columns,
    'Coefficient': model.coef_[0]
}).sort_values('Coefficient', ascending=False)

print("\nTop Positive Predictors (→ Survived):")
print(coef_df.head(5).to_string(index=False))

print("\nTop Negative Predictors (→ Not Survived):")
print(coef_df.tail(5).to_string(index=False))

Train size : 712 samples
Test  size : 179 samples

         LOGISTIC REGRESSION RESULTS
  Test  Accuracy  : 0.8380  (83.80%)
  ROC-AUC Score   : 0.8776
  CV Accuracy     : 0.8249 ± 0.0197  (5-fold)

Classification Report:
              precision    recall  f1-score   support

Not Survived       0.85      0.89      0.87       110
    Survived       0.81      0.75      0.78        69

    accuracy                           0.84       179
   macro avg       0.83      0.82      0.83       179
weighted avg       0.84      0.84      0.84       179

Confusion Matrix:
                      Pred: Not Survived  Pred: Survived
Actual: Not Survived                  98              12
Actual: Survived                      17              52

Top Positive Predictors (→ Survived):
          Feature  Coefficient
     Title_Master     1.344292
Family_size_Alone     0.720128
Family_size_Small     0.656502
        Title_Mrs     0.618514
             Fare     0.541932

Top Negative Predictors (→ Not Survi